# VAE latent walk

Use a prepared two-dimensional MNIST VAE to expose encoding, reparameterisation, decoding, and latent interpolation.

**Classroom pattern:** predict first, run one cell, inspect the intermediate state, then change one parameter. All random seeds are fixed for the first run.

## Before you run

- Use the environment from `README.md`; no lecturer-specific Conda environment is required.
- Set `LECTURE2_QUICK=1` before starting Jupyter for the CPU-safe verification path.
- CUDA is selected first, then Apple MPS, then CPU. A separate MLX installation is not required.
- Read the prediction prompt before each experiment. The fixed-seed result is a mechanism check, not a quality benchmark.
- Every generated file stays inside `fallbacks/` or `checkpoints/`. No cell uploads or publishes anything.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "demo_models.py").exists():
    candidate = ROOT / "lecture 2" / "demos"
    if candidate.exists():
        ROOT = candidate
sys.path.insert(0, str(ROOT))
(ROOT / "fallbacks").mkdir(exist_ok=True)
print(f"Demo root: {ROOT}")

import numpy as np
import matplotlib.pyplot as plt
import torch
from demo_models import TinyVAE
torch.manual_seed(7)

## 1. Load the small offline checkpoint

No training occurs during class. The checkpoint was trained locally on MNIST with a two-dimensional latent space.

In [ ]:
checkpoint = torch.load(ROOT / "checkpoints" / "vae_mnist_latent2.pt", map_location="cpu", weights_only=True)
model = TinyVAE(latent_dim=checkpoint["latent_dim"])
model.load_state_dict(checkpoint["state_dict"])
model.eval()
demo = np.load(ROOT / "data" / "mnist_demo_samples.npz")
images = torch.tensor(demo["images"], dtype=torch.float32)
labels = demo["labels"]
print("training loss, last epoch:", round(checkpoint["history"]["loss"][-1], 3))
print("available fixed examples:", labels.tolist())

## 2. Encoder output and reconstruction

The encoder produces μ and log σ². The decoder reconstructs a sample z. For a stable live comparison, this cell decodes μ directly.

In [ ]:
with torch.no_grad():
    mu, logvar = model.encode(images)
    recon = model.decode(mu)

fig, axes = plt.subplots(2, 10, figsize=(13, 3.1))
for i in range(10):
    axes[0, i].imshow(images[i, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, i].imshow(recon[i, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(str(labels[i]))
    axes[0, i].axis("off"); axes[1, i].axis("off")
axes[0, 0].set_ylabel("data"); axes[1, 0].set_ylabel("decode μ")
fig.suptitle("A two-dimensional bottleneck forces a smooth but imperfect representation")
fig.tight_layout()
fig.savefig(ROOT / "fallbacks" / "02_vae_reconstructions.png", dpi=180, bbox_inches="tight")
plt.show()

## 3. Reparameterisation with concrete numbers

In [ ]:
index = 7
epsilon = torch.tensor([[-0.5, 0.8]])
sigma = torch.exp(0.5 * logvar[index:index+1])
z = mu[index:index+1] + sigma * epsilon
print("digit:", labels[index])
print("mu:     ", mu[index].numpy().round(3))
print("sigma:  ", sigma[0].numpy().round(3))
print("epsilon:", epsilon[0].numpy())
print("z:      ", z[0].numpy().round(3))

## 4. Latent interpolation

Predict where identity becomes ambiguous. Linear interpolation may pass through a low-density region even when both endpoints reconstruct well.

In [ ]:
left, right = 1, 7
weights = torch.linspace(0, 1, 9)[:, None]
z_path = (1 - weights) * mu[left] + weights * mu[right]
with torch.no_grad():
    decoded = model.decode(z_path)

fig, axes = plt.subplots(1, 9, figsize=(12, 1.8))
for i, ax in enumerate(axes):
    ax.imshow(decoded[i, 0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"α={weights[i,0]:.2f}", fontsize=8)
    ax.axis("off")
fig.suptitle(f"Decode the line between digit {labels[left]} and digit {labels[right]}")
fig.tight_layout()
fig.savefig(ROOT / "fallbacks" / "02_vae_latent_walk.png", dpi=180, bbox_inches="tight")
plt.show()
print("endpoint coordinates:", mu[left].numpy().round(2), "→", mu[right].numpy().round(2))

## Interpretation check and optional extension

Identify where randomness enters, and explain why decoding the mean is useful for comparison but is not a posterior sample. Find an interpolation that crosses a sparse latent region and record the visual failure.

**Optional extension:** compare linear interpolation with spherical interpolation and measure reconstruction error along both paths.

Record the changed parameter, the fixed controls, the observed evidence, and one limitation.